# Lab 3.4: Plan Mode vs Direct Execution

**What you'll build:** A complexity assessment system that determines when to use plan mode vs direct execution -- and learn why jumping to implementation on complex tasks causes mistakes.

**Estimated time:** 20-25 minutes

| Phase | What happens | Time |
|-------|-------------|------|
| 1 | The Wrong Way -- direct execution on a complex multi-file task | 5 min |
| 2 | The Right Way -- plan first, then execute after review | 5 min |
| 3 | Your Turn -- build a complexity classifier that picks the right mode | 10 min |
| 4 | Stress Test -- validate your classifier across diverse scenarios | 5 min |

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

## The Setup

Developers ask Claude Code to perform tasks of varying complexity. Some tasks are simple single-file fixes; others are complex multi-file refactorings. Choosing the wrong mode leads to either wasted time (plan mode for a trivial fix) or missed files and inconsistencies (direct execution for a complex migration).

We'll build a system that assesses task complexity and recommends the right mode.

In [ ]:
# Task scenarios with ground truth
TASKS = [
    {
        "id": 1,
        "description": "Fix null pointer exception on line 42 of src/api/users.ts. Stack trace shows user.email is undefined when user lookup returns null.",
        "correct_mode": "direct",
        "reason": "Single file, clear stack trace, obvious fix (add null check)",
        "signals": {"files": 1, "clarity": "high", "familiarity": "known", "risk": "low"}
    },
    {
        "id": 2,
        "description": "Refactor the authentication system from session-based to JWT tokens. Auth logic exists in middleware, route handlers, user model, and 12 test files.",
        "correct_mode": "plan",
        "reason": "15+ files, architectural decision, multiple valid approaches",
        "signals": {"files": 15, "clarity": "low", "familiarity": "unknown", "risk": "high"}
    },
    {
        "id": 3,
        "description": "Add input validation to the create_user endpoint. Validate email format, password strength, and username uniqueness.",
        "correct_mode": "direct",
        "reason": "Single endpoint, well-defined requirements, localized change",
        "signals": {"files": 1, "clarity": "high", "familiarity": "known", "risk": "low"}
    },
    {
        "id": 4,
        "description": "Migrate the database from MySQL to PostgreSQL. Includes schema changes, ORM updates, query syntax changes, and migration scripts across the entire backend.",
        "correct_mode": "plan",
        "reason": "Entire backend affected, database migration risks, needs exploration first",
        "signals": {"files": 30, "clarity": "medium", "familiarity": "unknown", "risk": "high"}
    },
    {
        "id": 5,
        "description": "Fix typo in error message: 'Authetication failed' should be 'Authentication failed' in src/auth/errors.ts.",
        "correct_mode": "direct",
        "reason": "Single line change, zero risk, no exploration needed",
        "signals": {"files": 1, "clarity": "high", "familiarity": "known", "risk": "low"}
    },
    {
        "id": 6,
        "description": "Add a new microservice for payment processing. Needs new API endpoints, database schema, message queue integration, and deployment config.",
        "correct_mode": "plan",
        "reason": "Greenfield service, architectural decisions, multiple integration points",
        "signals": {"files": 20, "clarity": "low", "familiarity": "unknown", "risk": "high"}
    },
    {
        "id": 7,
        "description": "Execute the approved plan from yesterday: add rate limiting middleware to all API endpoints using the express-rate-limit package.",
        "correct_mode": "direct",
        "reason": "Plan already approved, implementation details are clear, following up on prior decision",
        "signals": {"files": 5, "clarity": "high", "familiarity": "known", "risk": "low"}
    },
    {
        "id": 8,
        "description": "Investigate why the CI pipeline takes 45 minutes. Could be test parallelization, Docker layer caching, dependency installation, or build configuration.",
        "correct_mode": "plan",
        "reason": "Investigation with unknown root cause, needs exploration before any changes",
        "signals": {"files": "unknown", "clarity": "low", "familiarity": "unknown", "risk": "medium"}
    }
]

print("=== TASK SCENARIOS ===")
for t in TASKS:
    print(f"\n  Task {t['id']}: {t['description'][:80]}...")
    print(f"  Correct mode: {t['correct_mode'].upper()}")
    print(f"  Reason: {t['reason']}")

---

## Phase 1: The Wrong Approach

Let's see what happens when we use direct execution on a complex task.

In [ ]:
# Simulate direct execution on the JWT migration (Task 2)
DIRECT_PROMPT = """You are Claude Code in direct execution mode. Implement this task immediately 
without planning or exploration:

Task: Refactor the authentication system from session-based to JWT tokens.

The codebase has:
- src/middleware/auth.ts (session validation)
- src/routes/login.ts (creates sessions)
- src/routes/logout.ts (destroys sessions)
- src/models/session.ts (session model)
- src/models/user.ts (user model with session references)
- 12 test files that mock sessions

Return a JSON object with:
- "files_modified": list of files you would change
- "approach": brief description of your implementation approach
- "potential_issues": things you might miss without planning
- "confidence": 1-10 how confident you are this is complete

Return ONLY the JSON object."""

response = client.messages.create(
    model=MODEL,
    max_tokens=1500,
    messages=[{"role": "user", "content": DIRECT_PROMPT}]
)

raw = response.content[0].text.strip()
if raw.startswith("```"):
    raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
direct_result = json.loads(raw)

print("=== DIRECT EXECUTION RESULT ===")
print(f"Files identified: {len(direct_result.get('files_modified', []))}")
print(f"Confidence: {direct_result.get('confidence', '?')}/10")
print(f"\nApproach: {direct_result.get('approach', 'N/A')}")
print(f"\nPotential issues (things that would be missed):")
for issue in direct_result.get('potential_issues', []):
    print(f"  - {issue}")

---

## Phase 2: The Right Approach

Now let's use plan mode: explore first, then propose an approach for human review.

In [ ]:
# Simulate plan mode on the same task
PLAN_PROMPT = """You are Claude Code in plan mode. Your job is to EXPLORE and PLAN, not implement.

Task: Refactor the authentication system from session-based to JWT tokens.

The codebase has:
- src/middleware/auth.ts (session validation)
- src/routes/login.ts (creates sessions)
- src/routes/logout.ts (destroys sessions)
- src/models/session.ts (session model)
- src/models/user.ts (user model with session references)
- 12 test files that mock sessions
- src/config/database.ts (session store config)
- src/utils/cookies.ts (session cookie handling)
- package.json (express-session dependency)

Phase 1 - EXPLORE: Identify ALL affected files and dependencies.
Phase 2 - ANALYZE: List multiple valid approaches with tradeoffs.
Phase 3 - PROPOSE: Recommend one approach with step-by-step plan.

Return a JSON object with:
- "affected_files": complete list of files that need changes
- "dependencies_to_add": new packages needed
- "dependencies_to_remove": packages no longer needed  
- "approaches": list of 2-3 valid approaches with pros/cons
- "recommended_approach": which approach and why
- "implementation_steps": ordered list of steps
- "risks": things to watch out for
- "confidence": 1-10

Return ONLY the JSON object."""

response = client.messages.create(
    model=MODEL,
    max_tokens=2500,
    messages=[{"role": "user", "content": PLAN_PROMPT}]
)

raw = response.content[0].text.strip()
if raw.startswith("```"):
    raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
plan_result = json.loads(raw)

print("=== PLAN MODE RESULT ===")
print(f"Files identified: {len(plan_result.get('affected_files', []))}")
print(f"Confidence: {plan_result.get('confidence', '?')}/10")

print(f"\nAffected files:")
for f in plan_result.get('affected_files', []):
    print(f"  - {f}")

print(f"\nApproaches considered:")
for i, approach in enumerate(plan_result.get('approaches', []), 1):
    if isinstance(approach, dict):
        print(f"  {i}. {approach.get('name', approach.get('approach', 'N/A'))}")
    else:
        print(f"  {i}. {approach}")

print(f"\nRecommended: {plan_result.get('recommended_approach', 'N/A')}")

print(f"\nImplementation steps:")
for i, step in enumerate(plan_result.get('implementation_steps', []), 1):
    print(f"  {i}. {step}")

In [ ]:
# Compare the two approaches
direct_files = len(direct_result.get('files_modified', []))
plan_files = len(plan_result.get('affected_files', []))

print("=== COMPARISON: Direct vs Plan ===")
print(f"{'Metric':<30} {'Direct':>12} {'Plan':>12}")
print(f"{'-'*30} {'-'*12} {'-'*12}")
print(f"{'Files identified':<30} {direct_files:>12} {plan_files:>12}")
print(f"{'Approaches considered':<30} {'1':>12} {len(plan_result.get('approaches', [])):>12}")
print(f"{'Confidence':<30} {direct_result.get('confidence', '?'):>12} {plan_result.get('confidence', '?'):>12}")
print(f"{'Risks identified':<30} {len(direct_result.get('potential_issues', [])):>12} {len(plan_result.get('risks', [])):>12}")

if plan_files > direct_files:
    print(f"\nPlan mode found {plan_files - direct_files} more affected files.")
    print("These would have been missed in direct execution, causing inconsistencies.")

---

## Phase 3: Your Turn

Build a complexity classifier that, given a task description, recommends plan mode or direct execution.

In [ ]:
# =============================================================
# TODO: Build a prompt that classifies task complexity
# =============================================================
#
# Your prompt should:
# 1. Assess the task along 4 dimensions: scope, clarity, familiarity, risk
# 2. Recommend "plan" or "direct" mode
# 3. Explain the reasoning
#
# Think about:
# - What signals indicate plan mode? (multi-file, unclear approach, unknown code)
# - What signals indicate direct? (single-file, clear fix, known pattern)
# - What about edge cases? (approved plan being implemented, investigation tasks)

def classify_task(task_description: str) -> dict:
    """
    Classify a task as needing plan mode or direct execution.
    
    Returns a dict with:
    - mode: "plan" or "direct"
    - confidence: 1-10
    - reasoning: explanation
    - signals: dict of complexity signals assessed
    """
    # TODO: Write your classification prompt
    prompt = f"""Assess the following development task and recommend whether to use
plan mode or direct execution in Claude Code.

Task: {task_description}

# TODO: Add your assessment criteria here.
# Consider: scope (files affected), clarity, familiarity, risk
# Consider: edge cases like "implementing an approved plan" or "investigation"

Return a JSON object with:
- "mode": "plan" or "direct"
- "confidence": 1-10
- "reasoning": one sentence explanation
- "signals": {{"scope": "single/multi", "clarity": "high/low", "familiarity": "known/unknown", "risk": "high/low"}}

Return ONLY the JSON object."""
    
    response = client.messages.create(
        model=MODEL,
        max_tokens=500,
        messages=[{"role": "user", "content": prompt}]
    )
    
    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return json.loads(raw)

# Test on one task
test_result = classify_task(TASKS[0]['description'])
print(f"Task 1: {TASKS[0]['description'][:60]}...")
print(f"  Predicted: {test_result.get('mode', '?')}")
print(f"  Expected:  {TASKS[0]['correct_mode']}")
print(f"  Correct:   {test_result.get('mode') == TASKS[0]['correct_mode']}")

In [ ]:
# =============================================================
# Checker: validates your classifier across all tasks
# =============================================================

print("=== CLASSIFIER EVALUATION ===")
correct = 0
total = len(TASKS)

for task in TASKS:
    try:
        result = classify_task(task['description'])
        predicted = result.get('mode', '?')
        expected = task['correct_mode']
        match = predicted == expected
        if match:
            correct += 1
        
        status = 'CORRECT' if match else 'WRONG'
        print(f"\n  Task {task['id']}: [{status}]")
        print(f"    Predicted: {predicted} | Expected: {expected}")
        print(f"    Reasoning: {result.get('reasoning', 'N/A')}")
        if not match:
            print(f"    Ground truth: {task['reason']}")
    except Exception as e:
        print(f"\n  Task {task['id']}: [ERROR] {e}")

accuracy = correct / total
print(f"\n=== ACCURACY: {correct}/{total} ({accuracy:.0%}) ===")
if accuracy >= 0.875:  # 7/8 or better
    print("PASSED -- Your classifier correctly identifies complexity.")
else:
    print(f"NEEDS WORK -- Improve your assessment criteria. Missed {total - correct} tasks.")

---

## Phase 4: Stress Test

Run your classifier multiple times on ambiguous tasks to check consistency.

In [ ]:
# Run classifier 3 times on each task and check consistency
print("Running classifier 3x per task for consistency...\n")

for task in TASKS[:4]:  # Test first 4 for speed
    predictions = []
    for _ in range(3):
        try:
            result = classify_task(task['description'])
            predictions.append(result.get('mode', '?'))
        except Exception:
            predictions.append('error')
    
    consistent = len(set(predictions)) == 1
    all_correct = all(p == task['correct_mode'] for p in predictions)
    
    status = 'CONSISTENT + CORRECT' if consistent and all_correct else \
             'CONSISTENT but WRONG' if consistent else 'INCONSISTENT'
    print(f"  Task {task['id']}: {predictions} -> {status}")

print("\nIf inconsistent, your classification criteria may be ambiguous.")
print("Add more explicit decision rules to your prompt.")

### Key Takeaways

1. **Plan mode explores broadly.** It identifies all affected files and considers multiple approaches before committing.
2. **Direct execution focuses narrowly.** It implements immediately -- great for clear tasks, dangerous for complex ones.
3. **The plan-then-execute pattern** is a two-phase workflow: deliberate, then implement the approved plan.
4. **Key signals for plan mode:** multi-file scope, unclear approach, unfamiliar codebase, architectural decisions.
5. **Key signals for direct:** single-file fix, clear stack trace, approved plan, known pattern.
6. **An approved plan should be executed directly** -- re-planning would second-guess the approved approach.